In [67]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [68]:
import os
import zipfile

if not os.path.exists("/content/MELD_RAW/MELD.Raw"):
    with zipfile.ZipFile("/content/drive/MyDrive/MELD-RAW.zip", "r") as zip_ref:
        zip_ref.extractall("/content/MELD_RAW")


print("Dataset Ready!")

Dataset Ready!


In [69]:
import os

for root, dirs, files in os.walk("/content/MELD_RAW"):
    print(root)

/content/MELD_RAW
/content/MELD_RAW/MELD.Raw
/content/MELD_RAW/MELD.Raw/test
/content/MELD_RAW/MELD.Raw/test/output_repeated_splits_test
/content/MELD_RAW/MELD.Raw/dev
/content/MELD_RAW/MELD.Raw/dev/dev_splits_complete
/content/MELD_RAW/MELD.Raw/train
/content/MELD_RAW/MELD.Raw/train/train_splits


In [70]:
import pandas as pd

train_df = pd.read_csv("/content/MELD_RAW/MELD.Raw/train/train_sent_emo.csv")
dev_df   = pd.read_csv("/content/MELD_RAW/MELD.Raw/dev_sent_emo.csv")
test_df  = pd.read_csv("/content/MELD_RAW/MELD.Raw/test_sent_emo.csv")

print(train_df.shape)
print(dev_df.shape)
print(test_df.shape)

(9989, 11)
(1109, 11)
(2610, 11)


In [71]:
import numpy as np
import os

base = "/content/drive/MyDrive/DMIC2/embeddings"

dev_text = np.load(os.path.join(base, "dev_text_pca.npy"))
dev_audio = np.load(os.path.join(base, "dev_audio_pca.npy"))
dev_video = np.load(os.path.join(base, "dev_video_pca.npy"))

print(dev_text.shape)
print(dev_audio.shape)
print(dev_video.shape)

(1108, 256)
(1108, 128)
(1108, 16)


In [72]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

In [73]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [74]:
import pickle
import numpy as np

base = "/content/drive/MyDrive/DMIC2/embeddings"

with open("/content/drive/MyDrive/DMIC2/checkpoints/text_pca.pkl","rb") as f:
    text_pca_model = pickle.load(f)

In [75]:
test_text_embeddings = np.load(f"{base}/test_text_embeddings.npy")

print(test_text_embeddings.shape)

(2610, 768)


In [76]:
test_text_pca = text_pca_model.transform(test_text_embeddings)

print(test_text_pca.shape)

(2610, 256)


In [77]:
np.save(f"{base}/test_text_pca.npy", test_text_pca)

print("Saved Successfully")

Saved Successfully


In [78]:
base = "/content/drive/MyDrive/DMIC2/embeddings"

train_text = np.load(os.path.join(base, "train_text_pca.npy"))
train_audio = np.load(os.path.join(base, "train_audio_pca.npy"))
train_video = np.load(os.path.join(base, "train_video_pca.npy"))

dev_text = np.load(os.path.join(base, "dev_text_pca.npy"))
dev_audio = np.load(os.path.join(base, "dev_audio_pca.npy"))
dev_video = np.load(os.path.join(base, "dev_video_pca.npy"))

test_text = np.load(os.path.join(base, "test_text_pca.npy"))
test_audio = np.load(os.path.join(base, "test_audio_pca.npy"))
test_video = np.load(os.path.join(base, "test_video_pca.npy"))

print(train_text.shape)
print(train_audio.shape)
print(train_video.shape)

print(test_text.shape)
print(test_audio.shape)
print(test_video.shape)

(9988, 256)
(9988, 128)
(9988, 16)
(2610, 256)
(2610, 128)
(2610, 16)


In [79]:
print("Train")
print(train_text.shape)
print(train_audio.shape)
print(train_video.shape)

print("\nDev")
print(dev_text.shape)
print(dev_audio.shape)
print(dev_video.shape)

print("\nTest")
print(test_text.shape)
print(test_audio.shape)
print(test_video.shape)

Train
(9988, 256)
(9988, 128)
(9988, 16)

Dev
(1108, 256)
(1108, 128)
(1108, 16)

Test
(2610, 256)
(2610, 128)
(2610, 16)


In [80]:
import pickle

with open("/content/drive/MyDrive/DMIC2/checkpoints/text_pca.pkl", "rb") as f:
    text_pca_model = pickle.load(f)

with open("/content/drive/MyDrive/DMIC2/checkpoints/audio_pca.pkl", "rb") as f:
    audio_pca_model = pickle.load(f)

with open("/content/drive/MyDrive/DMIC2/checkpoints/video_pca.pkl", "rb") as f:
    video_pca_model = pickle.load(f)

print("PCA Models Loaded Successfully!")

PCA Models Loaded Successfully!


In [81]:
class ModalityMLP(nn.Module):

    def __init__(self, input_dim, hidden_dim=256):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, hidden_dim),

            nn.LayerNorm(hidden_dim),

            nn.ReLU(),

            nn.Dropout(0.3)

        )

    def forward(self, x):

        return self.network(x)

In [82]:
class ContextEncoder(nn.Module):

    def __init__(self, hidden_dim=256):

        super().__init__()

        encoder = nn.TransformerEncoderLayer(

            d_model=hidden_dim,

            nhead=8,

            dim_feedforward=512,

            dropout=0.1,

            batch_first=True

        )

        self.encoder = nn.TransformerEncoder(

            encoder,

            num_layers=2

        )

    def forward(self, x):

        x = x.unsqueeze(1)

        x = self.encoder(x)

        return x.squeeze(1)

In [83]:
class UnimodalPredictor(nn.Module):

    def __init__(self, hidden_dim=256, num_classes=3):

        super().__init__()

        self.classifier = nn.Sequential(

            nn.Linear(hidden_dim, 128),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(128, num_classes)

        )

    def forward(self, x):

        return self.classifier(x)

In [84]:
class ModalityImportanceRanking(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, text_logits, audio_logits, video_logits):

        text_score = torch.max(
            torch.softmax(text_logits, dim=1),
            dim=1
        )[0]

        audio_score = torch.max(
            torch.softmax(audio_logits, dim=1),
            dim=1
        )[0]

        video_score = torch.max(
            torch.softmax(video_logits, dim=1),
            dim=1
        )[0]

        scores = torch.stack(
            [text_score, audio_score, video_score],
            dim=1
        )

        return scores

In [85]:
def rank_modalities(scores):

    ranked_indices = torch.argsort(
        scores,
        dim=1,
        descending=True
    )

    return ranked_indices

In [86]:
class CrossAttentionBlock(nn.Module):

    def __init__(self, hidden_dim=256):

        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=8,
            batch_first=True
        )

        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, query, key, value):

        query = query.unsqueeze(1)
        key = key.unsqueeze(1)
        value = value.unsqueeze(1)

        output, _ = self.attention(
            query,
            key,
            value
        )

        output = self.norm(output + query)

        return output.squeeze(1)

In [87]:
def select_modalities(text_feat, audio_feat, video_feat, ranked_indices):

    modalities = [text_feat, audio_feat, video_feat]

    primary = []
    secondary = []
    tertiary = []

    for i in range(ranked_indices.size(0)):

        idx = ranked_indices[i]

        primary.append(modalities[idx[0]][i])
        secondary.append(modalities[idx[1]][i])
        tertiary.append(modalities[idx[2]][i])

    primary = torch.stack(primary)
    secondary = torch.stack(secondary)
    tertiary = torch.stack(tertiary)

    return primary, secondary, tertiary

In [88]:
class Gate(nn.Module):

    def __init__(self, hidden_dim=256):

        super().__init__()

        self.gate = nn.Sequential(

            nn.Linear(hidden_dim * 2, hidden_dim),

            nn.Sigmoid()

        )

    def forward(self, x1, x2):

        g = self.gate(
            torch.cat([x1, x2], dim=1)
        )

        return g * x1 + (1 - g) * x2

In [89]:
class CCAFStage1(nn.Module):

    def __init__(self, hidden_dim=256):

        super().__init__()

        self.cross = CrossAttentionBlock(hidden_dim)

        self.gate = Gate(hidden_dim)

    def forward(self, primary, secondary):

        attended = self.cross(
            primary,
            secondary,
            secondary
        )

        fused = self.gate(
            primary,
            attended
        )

        return fused

In [90]:
class CCAFStage1(nn.Module):

    def __init__(self, hidden_dim=256):

        super().__init__()

        self.cross = CrossAttentionBlock(hidden_dim)

        self.gate = Gate(hidden_dim)

    def forward(self, primary, secondary):

        attended = self.cross(
            primary,
            secondary,
            secondary
        )

        fused = self.gate(
            primary,
            attended
        )

        return fused

In [91]:
class InitialFusion(nn.Module):

    def __init__(self, hidden_dim=256):

        super().__init__()

        self.P_from_S = PairwiseCrossAttention(hidden_dim)
        self.P_from_T = PairwiseCrossAttention(hidden_dim)

        self.S_from_P = PairwiseCrossAttention(hidden_dim)
        self.S_from_T = PairwiseCrossAttention(hidden_dim)

        self.T_from_P = PairwiseCrossAttention(hidden_dim)
        self.T_from_S = PairwiseCrossAttention(hidden_dim)

    def forward(self, P, S, T):

        P = self.P_from_T(
                self.P_from_S(P, S),
                T
            )

        S = self.S_from_T(
                self.S_from_P(S, P),
                T
            )

        T = self.T_from_S(
                self.T_from_P(T, P),
                S
            )

        return P, S, T

In [92]:
class CascadedRefinement(nn.Module):

    def __init__(self, hidden_dim=256):

        super().__init__()

        self.PS = PairwiseCrossAttention(hidden_dim)
        self.PST = PairwiseCrossAttention(hidden_dim)

    def forward(self, P, S, T):

        # Stage 1: Primary + Secondary
        PS = self.PS(P, S)

        # Stage 2: Refined (P,S) + Tertiary
        PST = self.PST(PS, T)

        return PST

In [93]:
class FusionClassifier(nn.Module):

    def __init__(self, hidden_dim=256, num_classes=3):

        super().__init__()

        self.classifier = nn.Sequential(

            nn.Linear(hidden_dim, hidden_dim),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(hidden_dim, num_classes)

        )

    def forward(self, fused):

        return self.classifier(fused)

In [94]:
class ModalityProjection(nn.Module):

    def __init__(
        self,
        text_dim=256,
        audio_dim=128,
        video_dim=16,
        hidden_dim=256
    ):

        super().__init__()

        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.audio_proj = nn.Linear(audio_dim, hidden_dim)
        self.video_proj = nn.Linear(video_dim, hidden_dim)

    def forward(self, text, audio, video):

        text = self.text_proj(text)
        audio = self.audio_proj(audio)
        video = self.video_proj(video)

        return text, audio, video

In [95]:
class ModalityImportance(nn.Module):

    def __init__(self, hidden_dim=256):

        super().__init__()

        self.text_score = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

        self.audio_score = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

        self.video_score = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

        self.softmax = nn.Softmax(dim=1)

    def forward(self, text, audio, video):

        text_score = self.text_score(text)

        audio_score = self.audio_score(audio)

        video_score = self.video_score(video)

        scores = torch.cat(
            [
                text_score,
                audio_score,
                video_score
            ],
            dim=1
        )

        weights = self.softmax(scores)

        return weights

In [96]:
print("ModalityProjection :", "ModalityProjection" in globals())
print("ContextEncoder :", "ContextEncoder" in globals())
print("ModalityImportance :", "ModalityImportance" in globals())
print("InitialFusion :", "InitialFusion" in globals())
print("CascadedRefinement :", "CascadedRefinement" in globals())
print("FusionClassifier :", "FusionClassifier" in globals())

ModalityProjection : True
ContextEncoder : True
ModalityImportance : True
InitialFusion : True
CascadedRefinement : True
FusionClassifier : True


In [97]:
class DMIC2(nn.Module):

    def __init__(
        self,
        text_dim=256,
        audio_dim=128,
        video_dim=16,
        hidden_dim=256,
        num_classes=3
    ):

        super().__init__()

        # Projection
        self.project = ModalityProjection(
            text_dim,
            audio_dim,
            video_dim,
            hidden_dim
        )

        # Context Encoder
        self.context = ContextEncoder(hidden_dim)

        # Modality Importance
        self.importance = ModalityImportance(hidden_dim)

        # Initial Pairwise Fusion
        self.initial_fusion = InitialFusion(hidden_dim)

        # Cascaded Refinement
        self.refinement = CascadedRefinement(hidden_dim)

        # Final Classifier
        self.classifier = FusionClassifier(
            hidden_dim,
            num_classes
        )
        # Unimodal Heads
        self.text_head = UnimodalHead(hidden_dim, num_classes)
        self.audio_head = UnimodalHead(hidden_dim, num_classes)
        self.video_head = UnimodalHead(hidden_dim, num_classes)

    def forward(
        self,
        text,
        audio,
        video
    ):

        # Projection
        text, audio, video = self.project(
            text,
            audio,
            video
        )

        # Context Encoding
        text = self.context(text)
        audio = self.context(audio)
        video = self.context(video)

        # Importance Scores
        weights = self.importance(
            text,
            audio,
            video
        )

        # -----------------------------
# Sample-wise Dynamic Ranking
# -----------------------------

# weights -> (batch_size, 3)

        sorted_idx = torch.argsort(
          weights,
          dim=1,
          descending=True)

        modalities = torch.stack(
            [
                text,
                audio,
                video
            ],
            dim=1
        )
        batch_size = text.size(0)
        P = modalities[
            torch.arange(batch_size),
            sorted_idx[:, 0]
            ]

        S = modalities[
        torch.arange(batch_size),
        sorted_idx[:, 1]
        ]

        T = modalities[
        torch.arange(batch_size),
        sorted_idx[:, 2]
        ]

        # Initial Pairwise Fusion
        P, S, T = self.initial_fusion(
            P,
            S,
            T
        )

        # Cascaded Refinement
        fused = self.refinement(
            P,
            S,
            T
        )
        # Unimodal Predictions
        text_output = self.text_head(text)

        audio_output = self.audio_head(audio)

        video_output = self.video_head(video)

        # Classification
        output = self.classifier(
            fused
        )

        return (
                output,
                text_output,
                audio_output,
                video_output,
                weights
                )

In [98]:
classes = [
    "ModalityProjection",
    "ContextEncoder",
    "CrossAttentionBlock",
    "Gate",
    "PairwiseCrossAttention",
    "InitialFusion",
    "CascadedRefinement",
    "ModalityImportance",
    "FusionClassifier",
    "DMIC2"
]

for cls in classes:
    print(f"{cls}: {cls in globals()}")

ModalityProjection: True
ContextEncoder: True
CrossAttentionBlock: True
Gate: True
PairwiseCrossAttention: True
InitialFusion: True
CascadedRefinement: True
ModalityImportance: True
FusionClassifier: True
DMIC2: True


In [99]:
class PairwiseCrossAttention(nn.Module):

    def __init__(self, hidden_dim=256):
        super().__init__()

        self.cross = CrossAttentionBlock(hidden_dim)
        self.gate = Gate(hidden_dim)

    def forward(self, target, source):

        attended = self.cross(
            target,
            source,
            source
        )

        fused = self.gate(
            target,
            attended
        )

        return fused

In [100]:
train_df = pd.read_csv("/content/MELD_RAW/MELD.Raw/train/train_sent_emo.csv")
dev_df = pd.read_csv("/content/MELD_RAW/MELD.Raw/dev_sent_emo.csv")
test_df = pd.read_csv("/content/MELD_RAW/MELD.Raw/test_sent_emo.csv")

label_encoder = LabelEncoder()
label_encoder.fit(train_df["Sentiment"])

train_labels = torch.LongTensor(
    label_encoder.transform(train_df["Sentiment"])[:9988]
)

dev_labels = torch.LongTensor(
    label_encoder.transform(dev_df["Sentiment"])[:1108]
)

test_labels = torch.LongTensor(
    label_encoder.transform(test_df["Sentiment"])[:2610]
)

print(train_labels.shape)
print(dev_labels.shape)
print(test_labels.shape)

torch.Size([9988])
torch.Size([1108])
torch.Size([2610])


In [101]:
from torch.utils.data import Dataset

class MELDDataset(Dataset):

    def __init__(self, text, audio, video, labels):

        self.text = text
        self.audio = audio
        self.video = video
        self.labels = labels

    def __len__(self):

        return len(self.labels)

    def __getitem__(self, idx):

        return (
            self.text[idx],
            self.audio[idx],
            self.video[idx],
            self.labels[idx]
        )

In [102]:
train_dataset = MELDDataset(
    train_text,
    train_audio,
    train_video,
    train_labels
)

dev_dataset = MELDDataset(
    dev_text,
    dev_audio,
    dev_video,
    dev_labels
)

test_dataset = MELDDataset(
    test_text,
    test_audio,
    test_video,
    test_labels
)

print(len(train_dataset))
print(len(dev_dataset))
print(len(test_dataset))

9988
1108
2610


In [103]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print(len(train_loader))
print(len(dev_loader))
print(len(test_loader))

313
35
82


In [104]:
class UnimodalHead(nn.Module):

    def __init__(self, hidden_dim=256, num_classes=3):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        # Global Average Pooling (Removed, as input is already a feature vector)
        # x = x.mean(dim=1)

        return self.classifier(x)

In [105]:
print("train_labels" in globals())

True


In [106]:
print("Train Text :", train_text.shape)
print("Train Audio:", train_audio.shape)
print("Train Video:", train_video.shape)

print("Train Labels:", train_labels.shape)

Train Text : (9988, 256)
Train Audio: (9988, 128)
Train Video: (9988, 16)
Train Labels: torch.Size([9988])


In [107]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [108]:
batch = next(iter(train_loader))

text, audio, video, labels = batch

print(text.shape)
print(audio.shape)
print(video.shape)
print(labels.shape)

torch.Size([32, 256])
torch.Size([32, 128])
torch.Size([32, 16])
torch.Size([32])


In [109]:
model = DMIC2().to(device)

batch = next(iter(train_loader))

text, audio, video, labels = batch

text = text.to(device)
audio = audio.to(device)
video = video.to(device)

outputs, text_out, audio_out, video_out, weights = model(
    text,
    audio,
    video
)

print(outputs.shape)
print(weights.shape)

torch.Size([32, 3])
torch.Size([32, 3])


In [110]:
model = DMIC2()

print(model)

DMIC2(
  (project): ModalityProjection(
    (text_proj): Linear(in_features=256, out_features=256, bias=True)
    (audio_proj): Linear(in_features=128, out_features=256, bias=True)
    (video_proj): Linear(in_features=16, out_features=256, bias=True)
  )
  (context): ContextEncoder(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=512, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
   

In [111]:
criterion_fused = nn.CrossEntropyLoss()

In [112]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DMIC2().to(device)

In [113]:
criterion_uni = nn.CrossEntropyLoss()

In [114]:
fusion_loss = criterion_fused(outputs, labels.to(device))

text_loss = criterion_uni(text_out, labels.to(device))
audio_loss = criterion_uni(audio_out, labels.to(device))
video_loss = criterion_uni(video_out, labels.to(device))

print("Fusion Loss :", fusion_loss.item())
print("Text Loss   :", text_loss.item())
print("Audio Loss  :", audio_loss.item())
print("Video Loss  :", video_loss.item())

Fusion Loss : 1.158975601196289
Text Loss   : 1.121151328086853
Audio Loss  : 1.0744799375534058
Video Loss  : 1.0041764974594116


In [115]:
uni_loss = text_loss + audio_loss + video_loss

print("Unimodal Loss :", uni_loss.item())

Unimodal Loss : 3.199807643890381


In [116]:
import torch.optim as optim

# Hyperparameters (from the paper)
alpha = 1.0
beta = 0.01
gamma = 0.2

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=1e-4
)

# Learning Rate Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2
)

print("Optimizer and Scheduler Created Successfully!")

Optimizer and Scheduler Created Successfully!


In [117]:
import torch.nn.functional as F

def importance_loss(weights, text_loss, audio_loss, video_loss):
    """
    Compute target modality importance from unimodal losses.
    Lower loss -> Higher importance.
    """

    losses = torch.tensor(
        [
            text_loss.detach(),
            audio_loss.detach(),
            video_loss.detach()
        ],
        device=weights.device
    )

    # Inverse loss (smaller loss = larger importance)
    target = 1.0 / (losses + 1e-8)

    # Normalize
    target = target / target.sum()

    # Repeat for whole batch
    target = target.unsqueeze(0).repeat(weights.size(0), 1)

    # MSE Loss
    loss = F.mse_loss(weights, target)

    return loss

In [118]:
imp_loss = importance_loss(
    weights,
    text_loss,
    audio_loss,
    video_loss
)

print("Importance Loss :", imp_loss.item())

Importance Loss : 0.0023847974371165037


In [119]:
def balance_loss(weights):
    """
    Encourage balanced use of all modalities.
    """

    target = torch.ones(3, device=weights.device) / 3

    batch_mean = weights.mean(dim=0)

    loss = F.mse_loss(batch_mean, target)

    return loss

In [120]:
bal_loss = balance_loss(weights)

print("Balance Loss :", bal_loss.item())

Balance Loss : 0.0017308169044554234


In [121]:
total_loss = (
    fusion_loss
    + alpha * imp_loss
    + beta * bal_loss
    + gamma * uni_loss
)
print("Total Loss :", total_loss.item())

Total Loss : 1.8013391494750977


In [122]:
def train_one_epoch(
    model,
    train_loader,
    optimizer,
    device
):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for text, audio, video, labels in train_loader:

        text = text.to(device)
        audio = audio.to(device)
        video = video.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs, text_out, audio_out, video_out, weights = model(
            text,
            audio,
            video
        )

        fusion_loss = criterion_fused(outputs, labels)

        text_loss = criterion_uni(text_out, labels)
        audio_loss = criterion_uni(audio_out, labels)
        video_loss = criterion_uni(video_out, labels)

        uni_loss = text_loss + audio_loss + video_loss

        imp_loss = importance_loss(
            weights,
            text_loss,
            audio_loss,
            video_loss
        )

        bal_loss = balance_loss(weights)

        total_loss = (
            fusion_loss
            + alpha * imp_loss
            + beta * bal_loss
            + gamma * uni_loss
        )

        total_loss.backward()

        optimizer.step()

        running_loss += total_loss.item()

        _, predicted = outputs.max(1)

        correct += predicted.eq(labels).sum().item()

        total += labels.size(0)

    epoch_loss = running_loss / len(train_loader)

    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [123]:
print(train_one_epoch)

<function train_one_epoch at 0x7e14c2685760>


In [124]:
@torch.no_grad()
def validate(
    model,
    val_loader,
    device
):

    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    for text, audio, video, labels in val_loader:

        text = text.to(device)
        audio = audio.to(device)
        video = video.to(device)
        labels = labels.to(device)

        outputs, text_out, audio_out, video_out, weights = model(
            text,
            audio,
            video
        )

        fusion_loss = criterion_fused(outputs, labels)

        text_loss = criterion_uni(text_out, labels)
        audio_loss = criterion_uni(audio_out, labels)
        video_loss = criterion_uni(video_out, labels)

        uni_loss = text_loss + audio_loss + video_loss

        imp_loss = importance_loss(
            weights,
            text_loss,
            audio_loss,
            video_loss
        )

        bal_loss = balance_loss(weights)

        total_loss = (
            fusion_loss
            + alpha * imp_loss
            + beta * bal_loss
            + gamma * uni_loss
        )

        running_loss += total_loss.item()

        _, predicted = outputs.max(1)

        correct += predicted.eq(labels).sum().item()

        total += labels.size(0)

    epoch_loss = running_loss / len(val_loader)

    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [125]:
print(validate)

<function validate at 0x7e14c26847c0>


In [ ]:
# ===========================================
# FINAL TRAINING LOOP
# ===========================================

num_epochs = 20

best_acc = 0.0

for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        device
    )

    val_loss, val_acc = validate(
        model,
        dev_loader,
        device
    )

    scheduler.step(val_loss)

    print("=" * 60)
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Acc  : {train_acc:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val Acc    : {val_acc:.4f}")

    if val_acc > best_acc:

        best_acc = val_acc

        torch.save(
            model.state_dict(),
            "best_dmic2_model.pth"
        )

        print("✅ Best Model Saved!")

print("\n🎉 Training Finished!")
print(f"Best Validation Accuracy : {best_acc:.4f}")

Epoch 1/20
Train Loss : 1.7037
Train Acc  : 0.4603
Val Loss   : 1.6490
Val Acc    : 0.4702
✅ Best Model Saved!
Epoch 2/20
Train Loss : 1.6759
Train Acc  : 0.4783
Val Loss   : 1.5998
Val Acc    : 0.5325
✅ Best Model Saved!
Epoch 3/20
Train Loss : 1.6555
Train Acc  : 0.4876
Val Loss   : 1.5980
Val Acc    : 0.5045
Epoch 4/20
Train Loss : 1.6452
Train Acc  : 0.5009
Val Loss   : 1.5708
Val Acc    : 0.5587
✅ Best Model Saved!
Epoch 5/20
Train Loss : 1.6309
Train Acc  : 0.5042
Val Loss   : 1.5702
Val Acc    : 0.5298
Epoch 6/20
Train Loss : 1.6176
Train Acc  : 0.5156
Val Loss   : 1.5471
Val Acc    : 0.5740
✅ Best Model Saved!
Epoch 7/20
Train Loss : 1.6065
Train Acc  : 0.5203
Val Loss   : 1.5705
Val Acc    : 0.5632
Epoch 8/20
Train Loss : 1.5955
Train Acc  : 0.5242
Val Loss   : 1.5968
Val Acc    : 0.5352
Epoch 9/20
Train Loss : 1.5833
Train Acc  : 0.5313
Val Loss   : 1.5689
Val Acc    : 0.5614
Epoch 10/20
Train Loss : 1.5615
Train Acc  : 0.5491
Val Loss   : 1.5682
Val Acc    : 0.5758
✅ Best Mo

In [ ]:
model.load_state_dict(torch.load("best_dmic2_model.pth", map_location=device))

print("✅ Best Model Loaded Successfully!")

In [128]:
from sklearn.metrics import accuracy_score, f1_score

@torch.no_grad()
def test(model, test_loader, device):

    model.eval()

    all_preds = []
    all_labels = []

    for text, audio, video, labels in test_loader:

        text = text.to(device)
        audio = audio.to(device)
        video = video.to(device)

        outputs, _, _, _, _ = model(text, audio, video)

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="weighted")

    return accuracy, f1, all_labels, all_preds

In [129]:
accuracy, f1, y_true, y_pred = test(
    model,
    test_loader,
    device
)

print(f"Test Accuracy : {accuracy:.4f}")
print(f"Weighted F1   : {f1:.4f}")

Test Accuracy : 0.6008
Weighted F1   : 0.5898


In [130]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    y_pred,
    digits=4
))

              precision    recall  f1-score   support

           0     0.5742    0.4970    0.5328       833
           1     0.6384    0.7675    0.6970      1256
           2     0.5013    0.3647    0.4222       521

    accuracy                         0.6008      2610
   macro avg     0.5713    0.5431    0.5507      2610
weighted avg     0.5906    0.6008    0.5898      2610



In [131]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_true,
    y_pred
)

print(cm)

[[414 324  95]
 [198 964  94]
 [109 222 190]]


In [132]:
torch.save(
    model.state_dict(),
    "DMIC2_Final_Model.pth"
)

print("✅ Final Model Saved")

✅ Final Model Saved
